# Ciclo invertido — métricas físicas nuevas (χ, Cv) | Kaggle

Pipeline: **parámetros → DDPM → imagen → Xception → parámetros**, con un tratamiento
físicamente fundamentado de la susceptibilidad **χ** y el calor específico **Cv**.

## Dos estimadores, dos ejes independientes
- **Fuente de datos:** original (1 imagen por punto de parámetros) vs generado (K imágenes/punto).
- **Estimador:**
  - **P (proxy de una sola configuración):** χ y Cv como el límite `q→0` del **factor de
    estructura** S(q) del campo conjugado (espín para χ, densidad de energía para Cv),
    extrapolado con la forma de **Ornstein–Zernike** `S(q)=S0/(1+q²ξ²)`. Fundamento:
    **teorema de fluctuación–disipación + ergodicidad espacial**.
  - **V (varianza de ensemble):** `χ = (N/T)·Var_k(M)`, `Cv = (N/T²)·Var_k(E)` sobre las K muestras.

## Tres cantidades por punto
| cantidad | datos | estimador |
|---|---|---|
| `*_orig_proxy` | original | **P** |
| `*_gen_proxy`  | generado (P por imagen, promediado sobre K) | **P** |
| `*_gen_ens`    | generado (ensemble) | **V** |

- **Comparación (R²):** `orig_proxy` vs `gen_proxy` → **mismo estimador P**, distinta fuente → justo.
- **Validación:** `gen_proxy` vs `gen_ens` → **misma fuente, P vs V** → comprueba que el proxy de
  una sola imagen reproduce la χ/Cv termodinámica (FDT/ergodicidad).

M y E usan la misma construcción: regime original = 1 imagen; lado generado = media sobre K.

> **TEST_FRACTION** (0–1) controla cuánto del split de test se evalúa: `1.0`=todo, `0.1`=10%.

> **Kaggle:** añade los datasets `dataset-spines-united-v2`, `weights-xception-model`,
> `weights-models` desde **+ Add Data**. Las rutas se autodetectan bajo `/kaggle/input`.


In [ ]:
import os, sys, math, time, glob, subprocess, gc
os.environ.setdefault("KERAS_BACKEND", "tensorflow")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

def _pip(pkg):
    try:
        __import__(pkg if pkg != "scikit-image" else "skimage")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
_pip("scikit-image")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from skimage.metrics import structural_similarity as ssim_fn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"TF {tf.__version__} | Torch {torch.__version__} | Device {DEVICE}")
for gpu in tf.config.list_physical_devices("GPU"):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except Exception: pass

# ── Autodetección de rutas en Kaggle ──────────────────────────────────────────
def find_file(basename, roots=("/kaggle/input", ".")):
    for root in roots:
        hits = glob.glob(f"{root}/**/{basename}", recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"No se encontró {basename} bajo {roots}")

DATASET_PATH         = find_file("dataset_unificado_v2.npz")
XCEPTION_INV_WEIGHTS = find_file("modelo_xception_fulldatabaseV3100.h5")
DDPM_CHECKPOINT      = find_file("ddpm_spines_final_39crop.pt")
for p in [DATASET_PATH, XCEPTION_INV_WEIGHTS, DDPM_CHECKPOINT]:
    print("[OK]", p)


In [ ]:
# ── Configuración ─────────────────────────────────────────────────────────────
PARAM_NAMES = ["T", "Jex2", "Jex3", "Jex4", "Kan1", "KanS", "Hex", "KDM"]
N_OUTPUTS   = 8
COND_DIM    = 8

# Splits (idénticos al entrenamiento original)
INV_SPLIT_SEED       = 42
INV_TEST_FRACTION    = 0.15
INV_VAL_FRACTION_REL = 0.1765
DDPM_SEED            = 42
DDPM_SUBSAMPLE_FRAC  = 1.0
DDPM_IMG_SIZE        = 40
TARGET_HW_INV        = (224, 224)
DDPM_FAST_STEPS      = 100
EPS_REL_FRAC         = 0.01

# ── Variable de control: fracción del SPLIT DE TEST a evaluar ─────────────────
#    1.0 = todo el test   ·   0.1 = 10% del test   ·   0.0 = nada
TEST_FRACTION = 0.10
BATCH_SEED    = 7

# Ensemble DDPM por punto (regime b)
K_ENS         = 16          # muestras generadas por punto de parámetros
PTS_PER_BATCH = 16          # puntos por llamada al DDPM (P*K imágenes/llamada)

# Factor de estructura / ajuste Ornstein–Zernike (estimador P)
RD_PIXELS     = 18.3
Q_FIT_MAX     = 2*np.pi/8.0  # ajustar OZ con longitudes de onda >= 8 px (régimen q->0)
SF_NBINS      = 24

assert 0.0 <= TEST_FRACTION <= 1.0, "TEST_FRACTION debe estar en [0,1]"
print(f"TEST_FRACTION={TEST_FRACTION} | K_ENS={K_ENS} | DDPM_FAST_STEPS={DDPM_FAST_STEPS}")


In [ ]:
# ── Dataset + scalers ─────────────────────────────────────────────────────────
data   = np.load(DATASET_PATH, mmap_mode="r")
imgs   = data["img"]
params = np.asarray(data["params"])
labels = np.asarray(data["labels"]) if "labels" in data.files else None
N = imgs.shape[0]
print(f"Dataset: {N:,} | imgs {imgs.shape} | params {params.shape}")

all_idx = np.arange(N)
idx_train_pool, idx_test_inv, _, _ = train_test_split(
    all_idx, params, test_size=INV_TEST_FRACTION, random_state=INV_SPLIT_SEED)
idx_train_inv, idx_val_inv, _, _ = train_test_split(
    idx_train_pool, params[idx_train_pool],
    test_size=INV_VAL_FRACTION_REL, random_state=INV_SPLIT_SEED)
scaler_inv = MinMaxScaler().fit(params[idx_train_inv])

rng = np.random.RandomState(DDPM_SEED)
sub_idx = rng.choice(N, size=int(N*DDPM_SUBSAMPLE_FRAC), replace=False)
params_sub_raw = params[sub_idx]
imgs_sub = np.asarray(imgs[sub_idx]).astype(np.float32)
idx_all_sub = np.arange(len(sub_idx))
idx_tr_sub, idx_temp_sub = train_test_split(idx_all_sub, test_size=0.30, random_state=DDPM_SEED)
scaler_ddpm = MinMaxScaler().fit(params_sub_raw[idx_tr_sub])
imgs_train_np = imgs_sub[idx_tr_sub].astype(np.float32)
mn, mx = float(imgs_train_np.min()), float(imgs_train_np.max())
print(f"Test split: {len(idx_test_inv):,} | mn={mn:.4f} mx={mx:.4f}")
del imgs_sub, imgs_train_np; gc.collect()


In [ ]:
from tensorflow.keras.applications import Xception
from tensorflow.keras.layers import (Input, GlobalAveragePooling2D, Dense,
                                     BatchNormalization, Dropout)
from tensorflow.keras.models import Model

def build_xception(n_outputs=8):
    inputs = Input(shape=(224, 224, 3), name="input_layer")
    base = Xception(weights=None, include_top=False, input_tensor=inputs)
    x = GlobalAveragePooling2D(name="global_average_pooling2d")(base.output)
    x = BatchNormalization(name="batch_normalization_4")(x)
    x = Dropout(0.4, name="dropout")(x)
    x = Dense(256, activation="relu", name="dense")(x)
    x = BatchNormalization(name="batch_normalization_5")(x)
    x = Dropout(0.3, name="dropout_1")(x)
    out = Dense(n_outputs, activation="linear", name="dense_1")(x)
    return Model(inputs, out, name="xception_inverso_V3100")

with tf.device("/cpu:0"):
    xception_model = build_xception()
    xception_model.load_weights(XCEPTION_INV_WEIGHTS)
print(f"Xception cargado en CPU. params={xception_model.count_params():,}")


In [ ]:
T_STEPS, BETA_START, BETA_END = 1000, 1e-4, 0.02

class DDPMScheduler:
    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02, schedule="linear", device="cpu"):
        self.T = T
        if schedule == "linear":
            betas = torch.linspace(beta_start, beta_end, T, device=device)
        elif schedule == "cosine":
            steps = T + 1; s = 0.008
            x = torch.linspace(0, T, steps, device=device)
            ac = torch.cos(((x/T)+s)/(1+s)*math.pi*0.5)**2; ac = ac/ac[0]
            betas = (1 - ac[1:]/ac[:-1]).clamp(max=0.999)
        else: raise ValueError(schedule)
        alphas = 1.0 - betas; ac = torch.cumprod(alphas, dim=0)
        self.betas=betas; self.alphas=alphas; self.alphas_cumprod=ac
        self.sqrt_alphas_cumprod=ac.sqrt(); self.sqrt_one_minus_alphas_cumprod=(1.0-ac).sqrt()

def sinusoidal_embedding(t, dim):
    half = dim//2
    freqs = torch.exp(-math.log(10000)*torch.arange(half, device=t.device).float()/(half-1))
    args = t[:,None].float()*freqs[None]
    return torch.cat([args.sin(), args.cos()], dim=-1)

class TimeCondEmbedding(nn.Module):
    def __init__(self, t_dim, cond_in, out_dim):
        super().__init__()
        self.t_mlp = nn.Sequential(nn.Linear(t_dim,out_dim), nn.SiLU(), nn.Linear(out_dim,out_dim))
        self.c_mlp = nn.Sequential(nn.Linear(cond_in,out_dim), nn.SiLU(), nn.Linear(out_dim,out_dim))
    def forward(self, t, cond):
        return self.t_mlp(sinusoidal_embedding(t, self.t_mlp[0].in_features)) + self.c_mlp(cond)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, emb_dim, groups=8, dropout=0.0):
        super().__init__()
        self.norm1=nn.GroupNorm(groups,in_ch);  self.conv1=nn.Conv2d(in_ch,out_ch,3,padding=1)
        self.norm2=nn.GroupNorm(groups,out_ch); self.conv2=nn.Conv2d(out_ch,out_ch,3,padding=1)
        self.emb_proj=nn.Linear(emb_dim,out_ch)
        self.dropout=nn.Dropout(dropout) if dropout>0 else nn.Identity()
        self.skip=nn.Conv2d(in_ch,out_ch,1) if in_ch!=out_ch else nn.Identity()
    def forward(self, x, emb):
        h=F.silu(self.norm1(x)); h=self.conv1(h)
        h=h+self.emb_proj(F.silu(emb))[:,:,None,None]
        h=F.silu(self.norm2(h)); h=self.dropout(h); h=self.conv2(h)
        return h+self.skip(x)

class SelfAttention(nn.Module):
    def __init__(self, ch, groups=8):
        super().__init__()
        self.norm=nn.GroupNorm(groups,ch); self.qkv=nn.Conv2d(ch,ch*3,1); self.proj=nn.Conv2d(ch,ch,1)
    def forward(self, x):
        B,C,H,W=x.shape; h=self.norm(x)
        q,k,v=self.qkv(h).chunk(3,dim=1)
        q=q.reshape(B,C,-1); k=k.reshape(B,C,-1); v=v.reshape(B,C,-1)
        attn=torch.softmax(torch.bmm(q.transpose(1,2),k)/math.sqrt(C),dim=-1)
        return x+self.proj(torch.bmm(v,attn.transpose(1,2)).reshape(B,C,H,W))

class ConditionalUNet(nn.Module):
    def __init__(self, img_channels=1, base_ch=64, ch_mults=(1,2,4), cond_dim=8, emb_dim=128, dropout=0.0):
        super().__init__()
        chs=[base_ch*m for m in ch_mults]
        self.emb=TimeCondEmbedding(t_dim=emb_dim, cond_in=cond_dim, out_dim=emb_dim)
        self.conv_in=nn.Conv2d(img_channels,chs[0],3,padding=1)
        self.down_blocks=nn.ModuleList(); self.down_samples=nn.ModuleList(); self.skip_channels=[]
        in_ch=chs[0]
        for i,out_ch in enumerate(chs):
            self.down_blocks.append(nn.ModuleList([ResBlock(in_ch,out_ch,emb_dim,dropout=dropout),
                                                   ResBlock(out_ch,out_ch,emb_dim,dropout=dropout)]))
            self.skip_channels.append(out_ch)
            self.down_samples.append(nn.Conv2d(out_ch,out_ch,4,stride=2,padding=1) if i<len(chs)-1 else nn.Identity())
            in_ch=out_ch
        self.mid_block1=ResBlock(chs[-1],chs[-1],emb_dim,dropout=dropout)
        self.mid_attn=SelfAttention(chs[-1]); self.mid_block2=ResBlock(chs[-1],chs[-1],emb_dim,dropout=dropout)
        self.up_blocks=nn.ModuleList(); self.up_samples=nn.ModuleList()
        for i,out_ch in enumerate(reversed(chs)):
            skip_ch=self.skip_channels[-(i+1)]
            self.up_blocks.append(nn.ModuleList([ResBlock(in_ch+skip_ch,out_ch,emb_dim,dropout=dropout),
                                                 ResBlock(out_ch,out_ch,emb_dim,dropout=dropout)]))
            self.up_samples.append(nn.ConvTranspose2d(out_ch,out_ch,4,stride=2,padding=1) if i<len(chs)-1 else nn.Identity())
            in_ch=out_ch
        self.norm_out=nn.GroupNorm(8,chs[0]); self.conv_out=nn.Conv2d(chs[0],img_channels,1)
    def forward(self, x, t, cond):
        emb=self.emb(t,cond); h=self.conv_in(x); skips=[]
        for (rb1,rb2),ds in zip(self.down_blocks,self.down_samples):
            h=rb1(h,emb); h=rb2(h,emb); skips.append(h); h=ds(h)
        h=self.mid_block1(h,emb); h=self.mid_attn(h); h=self.mid_block2(h,emb)
        for (rb1,rb2),us,sk in zip(self.up_blocks,self.up_samples,reversed(skips)):
            h=torch.cat([h,sk],dim=1); h=rb1(h,emb); h=rb2(h,emb); h=us(h)
        return self.conv_out(F.silu(self.norm_out(h)))

@torch.no_grad()
def fast_sample(model, cond, scheduler, n_steps=100, img_size=DDPM_IMG_SIZE):
    model.eval(); B=cond.shape[0]
    x=torch.randn(B,1,img_size,img_size,device=cond.device)
    timesteps=list(range(0,scheduler.T,scheduler.T//n_steps))[::-1]
    for t_val in timesteps:
        t_t=torch.full((B,),t_val,device=cond.device,dtype=torch.long)
        eps=model(x,t_t,cond)
        sa=scheduler.sqrt_alphas_cumprod[t_val]; s1a=scheduler.sqrt_one_minus_alphas_cumprod[t_val]
        x0=((x - s1a*eps)/sa).clamp(-1,1)
        if t_val>0:
            tp=max(t_val-scheduler.T//n_steps,0)
            x=scheduler.sqrt_alphas_cumprod[tp]*x0 + scheduler.sqrt_one_minus_alphas_cumprod[tp]*eps
        else:
            x=x0
    return x

print("Cargando DDPM...")
ckpt = torch.load(DDPM_CHECKPOINT, map_location=DEVICE, weights_only=False)
hp = ckpt["hyperparams"]; print("  hp:", hp)
ddpm_model = ConditionalUNet(img_channels=1, base_ch=hp["base_ch"], ch_mults=(1,2,4),
                             cond_dim=COND_DIM, emb_dim=hp["cond_emb_dim"], dropout=0.0).to(DEVICE)
state = ckpt["ema"] if ("ema" in ckpt and ckpt["ema"] is not None) else ckpt["model"]
ddpm_model.load_state_dict(state); ddpm_model.eval()
ddpm_scheduler = DDPMScheduler(T=T_STEPS, beta_start=BETA_START, beta_end=BETA_END,
                               schedule=hp["beta_schedule"], device=DEVICE)
print("  DDPM OK")


In [ ]:
def dataset_img_to_ddpm(img_39):
    x = np.asarray(img_39, dtype=np.float32)
    if x.ndim == 3: x = x[..., 0]
    x = np.pad(x, ((0,1),(0,1)), mode="reflect")          # 39->40
    x = (x - mn)/(mx - mn + 1e-8)
    return (x*2.0 - 1.0).astype(np.float32)

@torch.no_grad()
def generate_with_ddpm_batch(y_phys_batch, n_steps=DDPM_FAST_STEPS):
    cond = scaler_ddpm.transform(np.asarray(y_phys_batch, dtype=np.float32))
    cond_t = torch.tensor(cond, dtype=torch.float32, device=DEVICE)
    x = fast_sample(ddpm_model, cond_t, ddpm_scheduler, n_steps=n_steps, img_size=DDPM_IMG_SIZE)
    return x.detach().cpu().numpy()[:, 0]                  # (B,40,40)

XCEPTION_INFER_BATCH = 64
def infer_params_from_ddpm_batch(imgs_40, batch_size=XCEPTION_INFER_BATCH):
    out = []
    for s in range(0, len(imgs_40), batch_size):
        chunk = imgs_40[s:s+batch_size].astype(np.float32)[:, :39, :39]
        chunk = (chunk + 1.0)/2.0*(mx - mn) + mn
        chunk = chunk[..., None]
        chunk = tf.image.resize(chunk, TARGET_HW_INV)
        chunk = tf.image.grayscale_to_rgb(chunk)
        y_norm = xception_model.predict(chunk, verbose=0)
        out.append(scaler_inv.inverse_transform(y_norm))
    return np.concatenate(out, axis=0)
print("Inferencia configurada.")


In [ ]:
from IPython.display import display, HTML
plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"white","axes.grid":True,
                     "grid.color":"#cccccc","grid.linestyle":"--","grid.alpha":0.5,
                     "savefig.dpi":140,"savefig.bbox":"tight","font.family":"DejaVu Sans"})
COLOR_ORIG="#2c5f8d"; COLOR_GEN="#d77a3b"; COLOR_VAL="#2ca02c"

def _fmt(v, nd=4):
    if v is None: return "—"
    if isinstance(v,float) and (np.isnan(v) or np.isinf(v)): return "—"
    if isinstance(v,(int,np.integer)): return f"{int(v):d}"
    return f"{v:.{nd}f}"

def display_df_styled(df, title=None, num_cols=None, color_col=None, thresholds=(5,20), nd=4):
    cols=list(df.columns)
    if num_cols is None:
        num_cols=[c for c in cols if pd.api.types.is_numeric_dtype(df[c])]
    css="<style>.t{border-collapse:collapse;font-family:DejaVu Sans;font-size:12px}.t th{background:#2c5f8d;color:#fff;padding:6px 11px;border:1px solid #1d4666}.t td{padding:5px 11px;border:1px solid #dde2e6}.t tr:nth-child(even) td{background:#f5f7fa}.t td.num{text-align:right}.t td.good{color:#1b6f1b;font-weight:600}.t td.warn{color:#b45f06;font-weight:600}.t td.bad{color:#a30000;font-weight:600}.cap{color:#2c5f8d;font-weight:600;margin-top:8px}</style>"
    h=[css]
    if title: h.append(f'<p class="cap">{title}</p>')
    h.append('<table class="t"><thead><tr>'+''.join(f"<th>{c}</th>" for c in cols)+'</tr></thead><tbody>')
    for _,row in df.iterrows():
        h.append("<tr>")
        for c in cols:
            v=row[c]; cls="num" if c in num_cols else ""
            if color_col is not None and c==color_col:
                try:
                    av=abs(float(v))
                    cls += " good" if av<thresholds[0] else (" warn" if av<thresholds[1] else " bad")
                except Exception: pass
            h.append(f'<td class="{cls}">{_fmt(v,nd) if c in num_cols else v}</td>')
        h.append("</tr>")
    h.append("</tbody></table>"); display(HTML("".join(h)))
print("Estilo listo.")


---
## Métricas físicas — estimador P (factor de estructura, q→0) y estimador V (ensemble)

**Principio (FDT):** una función respuesta es el límite `q→0` del factor de estructura del
campo conjugado: `χ = S0_spin / T`, `Cv = S0_E / T²`, con `S(q)=S0/(1+q²ξ²)` (Ornstein–Zernike).

- **P (1 imagen):** `S(q)=|FFT(δφ)|²/N` del campo `φ` (espín → χ; densidad de energía local → Cv),
  promediado azimutalmente, y `S0=S(q→0)` extrapolado por ajuste OZ lineal `1/S vs q²`
  (se excluye `q=0`, trivial). Normalización `/N` ⇒ `S0` queda en la **misma escala** que la
  varianza de ensemble (por FDT), resolviendo el desajuste de escala del proxy anterior.
- **V (ensemble):** `χ=(N/T)Var_k(M)`, `Cv=(N/T²)Var_k(E)`.

**Cautelas (para el artículo):** la máscara de disco no es periódica → el FFT sufre fuga
espectral del borde (se aplica igual a original y generado, así que la comparación es
consistente); el `q` mínimo accesible es `2π/L` (L=40) ⇒ ξ fiable solo si `ξ ≲ L`; en fases
ordenadas (red de skyrmiones) excluir los picos de Bragg y ajustar el difuso a `q→0`.


In [ ]:
# ── Máscara de disco (sobre la imagen 40x40 del DDPM) ─────────────────────────
def circular_mask(H=DDPM_IMG_SIZE, W=DDPM_IMG_SIZE, rd=RD_PIXELS):
    cy, cx = (H-1)/2.0, (W-1)/2.0
    Y, X = np.ogrid[:H, :W]
    return ((Y-cy)**2 + (X-cx)**2) <= rd**2
DISK = circular_mask()
N_M  = int(DISK.sum())

# ── Imagen ────────────────────────────────────────────────────────────────────
def metric_mse(a,b):  return float(np.mean((a.astype(np.float64)-b.astype(np.float64))**2))
def metric_ssim(a,b): return float(ssim_fn(a,b,data_range=2.0))

# ── M y E (1 imagen) ──────────────────────────────────────────────────────────
def local_energy_density(img):
    s = img.astype(np.float64)*DISK
    nn = np.roll(s,1,0)+np.roll(s,-1,0)+np.roll(s,1,1)+np.roll(s,-1,1)
    return (-0.5*s*nn)*DISK                      # eps_i ; sum_i eps_i = -sum_bonds

def spin_magnetization(img): return float(img.astype(np.float64)[DISK].mean())
def energy_density(img):     return float(local_energy_density(img).sum()/N_M)

# ── Estimador P: factor de estructura S(q) + extrapolación OZ a q->0 ──────────
_qy = np.fft.fftfreq(DDPM_IMG_SIZE)*2*np.pi
_QY, _QX = np.meshgrid(_qy, _qy, indexing="ij")
_QMAG = np.sqrt(_QY**2 + _QX**2)

def _structure_factor_S0(field):
    """S0 = lim_{q->0} S(q) del campo (en el disco), por ajuste OZ lineal 1/S vs q^2."""
    s = field.astype(np.float64)
    sbar = s[DISK].mean()
    delta = (s - sbar)*DISK
    S2d = (np.abs(np.fft.fft2(delta))**2)/N_M
    q = _QMAG.ravel(); Sv = S2d.ravel()
    qmax = q.max(); edges = np.linspace(0, qmax, SF_NBINS+1)
    idx = np.clip(np.digitize(q, edges)-1, 0, SF_NBINS-1)
    qcen = np.zeros(SF_NBINS); Sbin = np.zeros(SF_NBINS); cnt = np.zeros(SF_NBINS)
    for b in range(SF_NBINS):
        m = idx==b
        if m.any(): qcen[b]=q[m].mean(); Sbin[b]=Sv[m].mean(); cnt[b]=m.sum()
    sel = (qcen>0) & (qcen<=Q_FIT_MAX) & (Sbin>1e-12) & (cnt>0)
    if sel.sum() >= 3:
        x = qcen[sel]**2; y = 1.0/Sbin[sel]
        A = np.vstack([np.ones_like(x), x]).T
        a, bcoef = np.linalg.lstsq(A, y, rcond=None)[0]
        if a > 0:
            return float(1.0/a)                  # S0 = 1/intercepto
    # fallback: valor de S en el bin de menor q positivo
    pos = (qcen>0) & (Sbin>0)
    return float(Sbin[pos][np.argmin(qcen[pos])]) if pos.any() else np.nan

def susceptibility_proxy(img, T):
    if T <= 0: return np.nan
    return _structure_factor_S0(img.astype(np.float64))/T

def specific_heat_proxy(img, T):
    if T <= 0: return np.nan
    return _structure_factor_S0(local_energy_density(img))/(T*T)

def metrics_P_single(img, T):
    """Estimador P sobre UNA imagen -> dict(M,E,chi,Cv)."""
    return dict(M=spin_magnetization(img), E=energy_density(img),
                chi=susceptibility_proxy(img, T), Cv=specific_heat_proxy(img, T))

# ── Estimador V: ensemble (regime b) ──────────────────────────────────────────
def chi_ensemble(M_list, T): return float(N_M*np.var(M_list)/T)    if (T>0 and len(M_list)>1) else np.nan
def Cv_ensemble(E_list, T):  return float(N_M*np.var(E_list)/(T*T)) if (T>0 and len(E_list)>1) else np.nan

print(f"Métricas listas. N_M={N_M} sitios en el disco. Q_FIT_MAX={Q_FIT_MAX:.4f} rad/px.")


In [ ]:
# ── Selección del subconjunto de test según TEST_FRACTION ─────────────────────
n_eval = int(round(TEST_FRACTION*len(idx_test_inv)))
if n_eval < 1:
    raise ValueError(f"TEST_FRACTION={TEST_FRACTION} -> 0 muestras. Sube el valor.")
rng_eval = np.random.RandomState(BATCH_SEED)
if n_eval >= len(idx_test_inv):
    eval_idx = np.sort(np.asarray(idx_test_inv, dtype=np.int64))
else:
    eval_idx = np.sort(rng_eval.choice(idx_test_inv, size=n_eval, replace=False))
N_pts = len(eval_idx)
y_in_phys = params[eval_idx].astype(np.float32)
print(f"Evaluando {N_pts}/{len(idx_test_inv)} puntos de test "
      f"({100*N_pts/len(idx_test_inv):.1f}%) | imágenes a generar: {N_pts*K_ENS:,}")


---
## Evaluación: generar K imágenes/punto y calcular las 3 cantidades
`*_orig_proxy` (P en la original) · `*_gen_proxy` (P por imagen, media sobre K) ·
`*_gen_ens` (V sobre el ensemble).


In [ ]:
PH = ["M","E","chi","Cv"]
orig_P = {k: np.full(N_pts, np.nan) for k in PH}     # estimador P, datos originales
gen_P  = {k: np.full(N_pts, np.nan) for k in PH}     # estimador P, generado (media sobre K)
gen_V  = {k: np.full(N_pts, np.nan) for k in PH}     # estimador V, generado (ensemble)
imgs_gen_rep = np.zeros((N_pts, DDPM_IMG_SIZE, DDPM_IMG_SIZE), dtype=np.float32)

t0 = time.time()
for ps in range(0, N_pts, PTS_PER_BATCH):
    pe = min(ps+PTS_PER_BATCH, N_pts)
    y_pts = y_in_phys[ps:pe]
    P = pe - ps
    y_rep = np.repeat(y_pts, K_ENS, axis=0)
    imgs_K = generate_with_ddpm_batch(y_rep).reshape(P, K_ENS, DDPM_IMG_SIZE, DDPM_IMG_SIZE)
    for j in range(P):
        k_pt = ps + j
        T_p = float(y_in_phys[k_pt, 0])
        # original (regime a) -> estimador P
        img_o = dataset_img_to_ddpm(np.asarray(imgs[eval_idx[k_pt]]))
        mo = metrics_P_single(img_o, T_p)
        for kk in PH: orig_P[kk][k_pt] = mo[kk]
        # generado: P por imagen + ensemble V
        gk = imgs_K[j]
        imgs_gen_rep[k_pt] = gk[0]
        Ms, Es, chis, Cvs = [], [], [], []
        for kk_img in range(K_ENS):
            m = metrics_P_single(gk[kk_img], T_p)
            Ms.append(m["M"]); Es.append(m["E"]); chis.append(m["chi"]); Cvs.append(m["Cv"])
        gen_P["M"][k_pt]=np.mean(Ms); gen_P["E"][k_pt]=np.mean(Es)
        gen_P["chi"][k_pt]=np.nanmean(chis); gen_P["Cv"][k_pt]=np.nanmean(Cvs)
        gen_V["M"][k_pt]=np.mean(Ms); gen_V["E"][k_pt]=np.mean(Es)
        gen_V["chi"][k_pt]=chi_ensemble(Ms, T_p); gen_V["Cv"][k_pt]=Cv_ensemble(Es, T_p)
    if (pe % (PTS_PER_BATCH*5) == 0) or pe == N_pts:
        el = time.time()-t0
        print(f"  {pe}/{N_pts} ({100*pe/N_pts:.0f}%)  {el/60:.1f} min  "
              f"ETA {el/max(pe,1)*(N_pts-pe)/60:.1f} min")
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f"Listo en {(time.time()-t0)/60:.1f} min")
print("Inferencia Xception sobre representantes...")
y_pred_phys = infer_params_from_ddpm_batch(imgs_gen_rep)
print("  y_pred:", y_pred_phys.shape)


In [ ]:
# ── R² de parámetros (entrada vs re-estimado por Xception) ────────────────────
rows = []
for i, name in enumerate(PARAM_NAMES):
    yt = y_in_phys[:, i]; yp = y_pred_phys[:, i]
    rng_p = yt.max()-yt.min()
    rows.append({"param":name, "MAE":mean_absolute_error(yt,yp),
                 "RMSE":mean_squared_error(yt,yp)**0.5,
                 "R2": r2_score(yt,yp) if np.var(yt)>1e-12 else np.nan,
                 "nMAE_%": 100*mean_absolute_error(yt,yp)/rng_p if rng_p>0 else np.nan})
df_reg = pd.DataFrame(rows)
display_df_styled(df_reg, title=f"Parámetros: entrada vs re-estimado  (n={N_pts})",
                  num_cols=["MAE","RMSE","R2","nMAE_%"], color_col="nMAE_%",
                  thresholds=(5,15), nd=4)


---
## Comparación física: `orig_proxy` vs `gen_proxy` (mismo estimador P)
M, E, χ, Cv con la **misma construcción** en ambos lados. Nota honesta: χ y Cv son de
segundo orden, así que su R² queda por debajo de M/E aunque el modelo sea perfecto (el techo
lo fija el ruido del proxy de una sola imagen en el eje original).


In [ ]:
labels_phys = {"M":"$M$","E":"$E$","chi":r"$\chi$","Cv":"$C_v$"}
rows = []
for k in PH:
    o = orig_P[k]; g = gen_P[k]
    v = np.isfinite(o) & np.isfinite(g)
    ov, gv = o[v], g[v]
    r2 = r2_score(ov, gv) if (len(ov)>=2 and np.var(ov)>1e-12) else np.nan
    rows.append({"métrica":{"chi":"χ","Cv":"Cv"}.get(k,k),
                 "mean(orig,P)":ov.mean() if len(ov) else np.nan,
                 "mean(gen,P)": gv.mean() if len(gv) else np.nan,
                 "MAE": mean_absolute_error(ov,gv) if len(ov) else np.nan,
                 "R²(orig_P→gen_P)": r2, "n": int(v.sum())})
display_df_styled(pd.DataFrame(rows),
    title="orig_proxy vs gen_proxy — mismo estimador P (comparación física correcta)",
    num_cols=["mean(orig,P)","mean(gen,P)","MAE","R²(orig_P→gen_P)","n"], nd=4)

fig, axes = plt.subplots(1, 4, figsize=(20, 4.7))
for ax, k in zip(axes, PH):
    o = orig_P[k]; g = gen_P[k]
    v = np.isfinite(o) & np.isfinite(g); ov, gv = o[v], g[v]
    ax.scatter(ov, gv, s=14, alpha=0.5, c=COLOR_ORIG, edgecolors="white", linewidths=0.3)
    if len(ov) >= 2:
        lo, hi = min(ov.min(),gv.min()), max(ov.max(),gv.max()); pad=0.05*(hi-lo+1e-9)
        ax.plot([lo-pad,hi+pad],[lo-pad,hi+pad],"k--",lw=1)
        r2 = r2_score(ov,gv) if np.var(ov)>1e-12 else float("nan")
        ax.set_title(f"{labels_phys[k]}   $R^2$={r2:.3f}   n={len(ov)}")
        ax.set_xlim(lo-pad,hi+pad); ax.set_ylim(lo-pad,hi+pad)
    ax.set_xlabel("original (P)"); ax.set_ylabel("generado (P, media K)")
fig.suptitle("Física: orig_proxy vs gen_proxy (mismo estimador P)")
plt.tight_layout(); plt.savefig("newmetrics_orig_vs_gen_proxy.png"); plt.show()


---
## Validación del proxy: `gen_proxy` (P) vs `gen_ens` (V) — misma fuente generada
Si P ≈ V sobre el set generado (donde ambos son calculables), el proxy de una sola imagen
queda **validado** como estimador de la χ/Cv termodinámica (FDT/ergodicidad). M y E son
idénticos por construcción (media = media), así que la validación se centra en χ y Cv.


In [ ]:
val_keys = ["chi","Cv"]
rows = []
for k in val_keys:
    p = gen_P[k]; vv = gen_V[k]
    m = np.isfinite(p) & np.isfinite(vv)
    pv, vvv = p[m], vv[m]
    r2 = r2_score(vvv, pv) if (len(pv)>=2 and np.var(vvv)>1e-12) else np.nan
    rows.append({"métrica":{"chi":"χ","Cv":"Cv"}.get(k,k),
                 "mean(gen,P)":pv.mean() if len(pv) else np.nan,
                 "mean(gen,V)":vvv.mean() if len(vvv) else np.nan,
                 "MAE": mean_absolute_error(vvv,pv) if len(pv) else np.nan,
                 "R²(V→P)": r2, "n": int(m.sum())})
display_df_styled(pd.DataFrame(rows),
    title="Validación: gen_proxy (P) vs gen_ens (V) sobre el set generado",
    num_cols=["mean(gen,P)","mean(gen,V)","MAE","R²(V→P)","n"], nd=4)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.7))
for ax, k in zip(axes, val_keys):
    p = gen_P[k]; vv = gen_V[k]
    m = np.isfinite(p) & np.isfinite(vv); pv, vvv = p[m], vv[m]
    ax.scatter(vvv, pv, s=14, alpha=0.5, c=COLOR_VAL, edgecolors="white", linewidths=0.3)
    if len(pv) >= 2:
        lo, hi = min(vvv.min(),pv.min()), max(vvv.max(),pv.max()); pad=0.05*(hi-lo+1e-9)
        ax.plot([lo-pad,hi+pad],[lo-pad,hi+pad],"k--",lw=1)
        r2 = r2_score(vvv,pv) if np.var(vvv)>1e-12 else float("nan")
        ax.set_title(f"{labels_phys[k]}   $R^2$(V→P)={r2:.3f}   n={len(pv)}")
        ax.set_xlim(lo-pad,hi+pad); ax.set_ylim(lo-pad,hi+pad)
    ax.set_xlabel("ensemble V  (N·Var/T...)"); ax.set_ylabel("proxy P  (media S0/T...)")
fig.suptitle("Validación del proxy P frente al ensemble V (datos generados)")
plt.tight_layout(); plt.savefig("newmetrics_proxy_validation.png"); plt.show()


In [ ]:
# ── Guardar resultados ────────────────────────────────────────────────────────
np.savez_compressed("cycle_newmetrics_results.npz",
    eval_idx=eval_idx, y_in_phys=y_in_phys, y_pred_phys=y_pred_phys,
    TEST_FRACTION=TEST_FRACTION, K_ENS=K_ENS,
    **{f"orig_P__{k}": orig_P[k] for k in PH},
    **{f"gen_P__{k}":  gen_P[k]  for k in PH},
    **{f"gen_V__{k}":  gen_V[k]  for k in PH})
df_reg.to_csv("cycle_newmetrics_params.csv", index=False)
print("Guardado: cycle_newmetrics_results.npz, cycle_newmetrics_params.csv")
print(f"Resumen: TEST_FRACTION={TEST_FRACTION} | n={N_pts} | K_ENS={K_ENS}")
